# LangChain: RAG & Q&A

## Outline
* مفهوم RAG
* ساخت knowledge base (loader → split → embed → store)
* 2-Step RAG با LCEL
* Agentic RAG با `create_agent`
* مقایسه دو روش


In [5]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())


In [7]:
# pip install langchain-openai langchain-community faiss-cpu

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


## ۱. لود PDF



In [9]:
# ── Step 1: Load (تغییر یافته برای PDF) ──

#pip install pypdf
from langchain_community.document_loaders import PyPDFLoader

# لود کردن فایل PDF
loader = PyPDFLoader("sample_doc.pdf")
docs = loader.load()

print(f"Loaded {len(docs)} pages from PDF")
# نمایش بخشی از صفحه اول برای تست
print(f"Sample content: {docs[0].page_content[:100]}...")

Ignoring wrong pointing object 105 0 (offset 0)


Loaded 6 pages from PDF
Sample content: ﺑﺎﺳﻣﮫ ﺗﻌﺎﻟﯽ    اﺻﻼح ﻧﺎﻣﮫ ﺷﯾوه اﺟراﯾﯽ  ﻧﺣوه  اﯾﺟﺎد  و  ﺗرﻣﯾم  ﺳﺎﺑﻘﮫ  ﺗﺣﺻﯾﻠﯽ  ھﺎی آزﻣون  ﻧﮭﺎﯾﯽ  دوره  ...


## ۲. ساخت Knowledge Base

In [11]:
# ── Step 2: Split ──
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
)
splits = splitter.split_documents(docs)
print(f"Split into {len(splits)} chunks")


Split into 19 chunks


In [13]:
# ── Step 3: Embed & Store ──
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")


In [15]:
# ساخت vector store
vectorstore = FAISS.from_documents(splits, embeddings)
print("Vector store created!")

# تست similarity search
query = "کی از متقاضیان ثبت نام انجام میگیرد؟"
results = vectorstore.similarity_search(query, k=2)
print(f"\nTop {len(results)} results for '{query}':")
for i, doc in enumerate(results):
    print(f"  {i+1}. {doc.page_content[:100]}")


Vector store created!

Top 2 results for 'کی از متقاضیان ثبت نام انجام میگیرد؟':
  1. در
 
ﺳﺎل ھﺎی
 
ﻣﺧﺗﻠف
 
ﻣﺗﻔﺎوت
 
ﺑﺎﺷد(.
 
ﺳﺎل
 
ﻣدﻧظر
 
در
 
ﻣﺣﺎﺳﺑﮫ
 
ﻧﻣره
 
ﺗراز،
 
ﺳﺎل
 
اﺧذ
 
آزﻣو
  2. دﺑﯾرﮐل ﺷورای ﻋﺎﻟﯽ  آﻣوزش  و  ﭘرورش    اﺻﻼﺣﺎت ﺷﯾوه ﻧﺎﻣﮫ  اﺟراﯾﯽ  ﻧﺣوه  اﯾﺟﺎد  ﺳﺎﺑﻘﮫ  ﺗﺣﺻﯾﻠﯽ  و  ﺗرﻣﯾم


In [17]:
import re

# تابعی برای حذف اینترهای اضافه و تمیزکاری متن
def clean_text(text):
    # حذف اینترهای اضافی (\n) و تبدیل آن‌ها به فاصله
    text = re.sub(r'\n+', ' ', text)
    # حذف فاصله‌های دو یا چندتایی و تبدیل به یک فاصله
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

loader = PyPDFLoader("sample_doc.pdf")
docs = loader.load()

for doc in docs:
    doc.page_content = clean_text(doc.page_content)

# حالا متن‌ها تمیز هستند و می‌توانید ادامه کار (Splitter) را انجام دهید
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, # کمی اندازه را بزرگتر کنید تا متن منسجم‌تر باشد
    chunk_overlap=100,
)
splits = splitter.split_documents(docs)

Ignoring wrong pointing object 105 0 (offset 0)


In [19]:
len(splits)

16

In [21]:
# ساخت vector store
vectorstore = FAISS.from_documents(splits, embeddings)
print("Vector store created!")

results = vectorstore.similarity_search(query, k=2)
print(f"\nTop {len(results)} results for '{query}':")
for i, doc in enumerate(results):
    print(f"  {i+1}. {doc.page_content}")
    print("-"*10)


Vector store created!

Top 2 results for 'کی از متقاضیان ثبت نام انجام میگیرد؟':
  1. دﺑﯾرﮐل ﺷورای ﻋﺎﻟﯽ آﻣوزش و ﭘرورش اﺻﻼﺣﺎت ﺷﯾوه ﻧﺎﻣﮫ اﺟراﯾﯽ ﻧﺣوه اﯾﺟﺎد ﺳﺎﺑﻘﮫ ﺗﺣﺻﯾﻠﯽ و ﺗرﻣﯾم ﺳﺎﺑﻘﮫ ﺗﺣﺻﯾﻠﯽ آزﻣون ھﺎی ﻧﮭﺎﯾﯽ دوره دوم ﻣﺗوﺳطﮫ ﻣورد ﺗﺄﯾﯾد اﺳت، ﺑﮫ ﻣورد اﺟرا ﮔذاﺷﺗﮫ ﺷود. ﻋﻠﯾرﺿﺎ ﮐﺎظﻣﯽ وزﯾر آﻣوزش و ﭘرورش 6
----------
  2. ﺗﺑﺻره 4: ﺑرای ﺗﺳﮭﯾل ﺛﺑت ﻧﺎم ﻣﺗﻘﺎﺿﯾﺎن اﯾﺟﺎد و ﯾﺎ ﺗرﻣﯾم ﺳﺎﺑﻘﮫ ﺗﺣﺻﯾﻠﯽ در ﺧﺎرج از ﮐﺷور، ﻣرﮐز اﻣور ﺑﯾن اﻟﻣﻠل و ﻣدارس ﺧﺎرج از ﮐﺷور ﻧﺳﺑت ﺑﮫ اﺧذ ﺗﻘﺎﺿﺎ از ﻣﺗﻘﺎﺿﯾﺎن ﺑﮫ طرﯾق ﻣﻘﺗﺿﯽ اﻗدام و در ﺑﺎزه زﻣﺎﻧﯽ اﻋﻼم ﺷده ﺑرای ﺛﺑت ﻧﺎم در ﺳﺎﻣﺎﻧﮫ ﻣرﺑوط ﺛﺑت ﻣﯽ ﻧﻣﺎﯾد. 6-ھزﯾﻧﮫ ﺷرﮐت در آزﻣون ھﺎی ﻧﮭﺎﯾﯽ ﺑرای اﯾﺟﺎد و ﯾﺎ ﺗرﻣﯾم ﺳﺎﺑﻘﮫ ﺗﺣﺻﯾﻠﯽ ﺑﮫ ﺗﻔﮑﯾﮏ ھر درس و ﻣطﺎﺑق ﻣﺻوﺑﮫ ھﯾﺋت وزﯾران اﺧذ ﻣﯽ ﺷود. (ث ﺳﺎﯾر ﺿواﺑط و ﻧﮑﺎت :ﺗوﺟﮫ ﻗﺎﺑل 1- ﻧﺣوه و ﻣﯾزان ﮐﺎرﺑرد ﻧﺗﺎﯾﺞ ھرﯾﮏ از ﺳواﻻت آزﻣون ھﺎی ﻧﮭﺎﯾﯽ در ﻓراﯾﻧدھﺎی ﺗوﻟﯾد ﺳﺎﺑﻘﮫ ﺗﺣﺻﯾﻠﯽ و ﻋﻣﻠﮑرد )ﮐﺎرﻧﺎﻣﮫ( ﺗﺣﺻﯾﻠﯽ ﺗوﺳط ﻣرﮐز ارزﺷﯾﺎﺑﯽ و ﺗﺿﻣﯾن ﮐﯾﻔﯾت ﻧظﺎم آﻣوزش و ﭘرورش ﺗﻌﯾﯾن و اﻋﻣﺎل ﻣﯽ ﺷود. 2- داﻧش آﻣوزان ﺷﺎﺧﮫ ﻧظری ﺷﺎﻏل ﺑﮫ ﺗﺣﺻﯾل در ﻣدارس ﺑزرﮔﺳﺎﻻن، آﻣوزش از راه دور، اﯾﺛﺎرﮔران و داوطﻠﺑﺎن آزاد ﻣ

## ۳. 2-Step RAG با LCEL

In [31]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# RAG prompt
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """Answer the question based only on the following context.
If you don't know the answer, say "نمیدانم. اطلاعی در این رابطه ندارم"

Context:
{context}"""),
("human", "{question}"),
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 2-Step RAG Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# تست
response = rag_chain.invoke("کی از متقاضیان ثبت نام انجام میگیرد؟")
print(response)


ثبت نام از متقاضیان ایجاد و یا ترمیم سابقه تحصیلی حداکثر تا 2 هفته قبل از برگزاری آزمون های نهایی خرداد و در بازه زمانی حداکثر 10 روزه انجام می‌شود.


In [25]:
# تست 2
response = rag_chain.invoke("نمره کل بر اساس چه بندی و چه مصوبه ای حساب میشه؟")
print(response)

نمره کل بر اساس بند 2 - 2 مصوبه جلسه 843 مورخ 15/04/1400 شورای عالی انقلاب فرهنگی محاسبه می‌شود.


In [27]:
# تست 2
response = rag_chain.invoke("نمره کل بر اساس چه فرمولی محاسبه میگردد؟")
print(response)

نمره کل بر اساس میانگین نمره وزنی نمرات ترازشده دروس عمومی و تخصصی محاسبه می‌شود.


In [33]:
# تست 2
response = rag_chain.invoke("درس شیمی مفید تر است یا ریاضی")
print(response)

نمیدانم. اطلاعی در این رابطه ندارم
